In [1]:
#This code takes the original data, the prediction of nns and the BMS, computes the rmse and mae and saves everything into a dataframe 

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import seaborn as sns
import matplotlib.gridspec as gridspec
import ast
import sys
sys.path.append('../no_degeneracy/')
sys.path.append('../no_degeneracy/Prior/')
from mcmc import *
from parallel import *
from fit_prior import read_prior_par
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error

In [2]:
def clean_index(dataframe):
    dataframe.set_index('Unnamed: 0', inplace=True)
    dataframe.index.name = None
    dataframe= dataframe.reset_index(drop=True)
    return dataframe

def add_bms_pred(dataframe, bms_trace, number_param):
    VARS = ['x1',]
    x = dn[[c for c in VARS]].copy()
    y=dataframe.noise

    if number_param==10:
        prior_par = read_prior_par('../no_degeneracy/Prior/final_prior_param_sq.named_equations.nv1.np10.2017-10-18 18:07:35.089658.dat')
    elif number_param==20:
        prior_par = read_prior_par('../no_degeneracy/Prior/final_prior_param_sq.named_equations.nv1.np20.maxs200.2024-05-10 162907.551306.dat')

    #mdl model
    minrow = bms_trace[bms_trace.H == min(bms_trace.H)].iloc[0]
    minH, minexpr, minparvals = minrow.H, minrow.expr, ast.literal_eval(minrow.parvals)
    t = Tree(
        variables=list(x.columns),
        parameters=['a%d' % i for i in range(number_param)],
        x=x, y=y,
        prior_par=prior_par,
        max_size=200,
        from_string=minexpr,
    )

    t.set_par_values(deepcopy(minparvals))

    dplot = deepcopy(dn)
    dplot['ybms'] = t.predict(x)

    return dplot
    

In [4]:
#Read NN and BMS data
functions=['leaky_ReLU', 'tanh'] #tanh, leaky_ReLU
realizations=2
N=9

sigmas=[0.0, 0.02, 0.04,0.06, 0.08, 0.10, 0.12, 0.14, 0.16, 0.18, 0.20]
resolution='1x' #0.5x, 1x, 2x, 4e-3x
resolutions={'0.5x':'0.1', '1x':'0.05' , '2x': '0.025' , '4e-3x':'0.004' }

NPAR=10 #10, 20
steps=50000



rmse_nn_train=[];rmse_nn_test=[]
rmse_mdl_train=[];rmse_mdl_test=[]

mae_nn_train=[];mae_nn_test=[]
mae_mdl_train=[];mae_mdl_test=[]

n_index=[];r_index=[];sigma_index=[];function_index=[]

#Put mae and rmse of each simulation (on nn and bms) in a dataframe
for function in functions:

    for sigma in sigmas:

        for r in range(realizations+1):

            file_model='NN_no_overfit_' + function + '_sigma_' + str(sigma) + '_r_' + str(r) + '.csv'
            model_d='../../data/nns/' + resolution + '_resolution/approximation/' + file_model
            d=pd.read_csv(model_d)

            n_points=int(len(d.index)/10)
            train_fraction=3/4;train_size=int(n_points*train_fraction)
            

            for n in range(N+1):
                n_index.append(n);r_index.append(r);sigma_index.append(sigma);function_index.append(function)
            
                dn=d[d['rep']==n]
                dn=clean_index(dn)

                #Read BMS data
                filename='BMS_'+function+'_n_'+str(n)+'_sigma_'+str(sigma)+ '_r_' + str(r) + '_res_' + resolutions[resolution] + '_trace_'\
                +str(steps)+'_prior_'+str(NPAR)+ '.csv'

                print(function, sigma, n, r)
                
                try:
                    print("hello")
                    trace=pd.read_csv('../../data/MSTraces/' + resolution + '_resolution/' + filename, sep=';', header=None,
                                      names=['t','H','expr','parvals','kk1','kk2','kk3'])
                    print("bye")
                    dplot=add_bms_pred(dn, trace, NPAR)
                except FileNotFoundError:
                    dplot = deepcopy(dn) #If no bms errors available, fill the dataframe with zeros
                    dplot['ybms'] = [0] * len(dplot)
                

                #Compute errors
                #-----------------------------------------------------------------------------------------------------------------------
                #nns
                rmse_nn_train_i=root_mean_squared_error(dplot.loc[:train_size-1]['ymodel'],dplot.loc[:train_size -1]['y'])
                rmse_nn_train.append(rmse_nn_train_i)
            
                rmse_nn_test_i=root_mean_squared_error(dplot.loc[train_size-1:]['ymodel'],dplot.loc[train_size -1:]['y'])
                rmse_nn_test.append(rmse_nn_test_i)

                mae_nn_train_i=mean_absolute_error(dplot.loc[:train_size-1]['ymodel'],dplot.loc[:train_size -1]['y'])
                mae_nn_train.append(mae_nn_train_i)
            
                mae_nn_test_i=mean_absolute_error(dplot.loc[train_size-1:]['ymodel'],dplot.loc[train_size -1:]['y'])
                mae_nn_test.append(mae_nn_test_i)

                
                #bms
                try:
                    rmse_mdl_train_i=root_mean_squared_error(dplot.loc[:train_size-1]['ybms'],dn.loc[:train_size-1]['y'])
                except ValueError:
                    rmse_mdl_train_i=np.inf
                rmse_mdl_train.append(rmse_mdl_train_i)

                try:
                    rmse_mdl_test_i=root_mean_squared_error(dplot.loc[train_size-1:]['ybms'],dn.loc[train_size-1:]['y'])
                except (ValueError, RuntimeWarning) as e:
                    rmse_mdl_test_i=np.inf
                
                rmse_mdl_test.append(rmse_mdl_test_i)

                try:
                    mae_mdl_train_i=mean_absolute_error(dplot.loc[:train_size-1]['ybms'],dplot.loc[:train_size -1]['y'])
                except ValueError:
                    mae_mdl_train_i=np.inf
                mae_mdl_train.append(mae_mdl_train_i)

                try:
                    mae_mdl_test_i=mean_absolute_error(dplot.loc[train_size-1:]['ybms'],dplot.loc[train_size -1:]['y'])
                except ValueError:
                    mae_mdl_test_i=np.inf
                
                mae_mdl_test.append(mae_mdl_test_i)
                #-----------------------------------------------------------------------------------------------------------------------


#Save all in a dataframe
errors_df=pd.DataFrame({'sigma':sigma_index, 'function':function_index, 'mae_nn_train':mae_nn_train, 'mae_nn_test':mae_nn_test, 'mae_mdl_train':mae_mdl_train, 
                        'mae_mdl_test':mae_mdl_test, 'rmse_nn_train':rmse_nn_train, 'rmse_nn_test': rmse_nn_test, 
                        'rmse_mdl_train':rmse_mdl_train, 'rmse_mdl_test': rmse_mdl_test, 'n':n_index, 'r': r_index})
errors_df.to_csv('../../data/test_errors_median_' + resolution + '.csv')
display(errors_df)

leaky_ReLU 0.0 0 0
hello
bye


<lambdifygenerated-11561>:2: RuntimeWarning: invalid value encountered in power
  return x1*tan(x1*(x1 + x1**x1))**2
<lambdifygenerated-11562>:2: RuntimeWarning: invalid value encountered in power
  return x1*tan(x1*(x1 + x1**x1))**2
<lambdifygenerated-11565>:2: RuntimeWarning: invalid value encountered in power
  return x1*tan(x1*(_a4_**(x1**x1) + x1))**2
/export/home/shared/Projects/ANN/Sergio/BMS_approximator/bin/ann_functions/../no_degeneracy/mcmc.py:654: OptimizeWarning: Covariance of the parameters could not be estimated
  res = curve_fit(
<lambdifygenerated-11566>:2: RuntimeWarning: invalid value encountered in power
  return x1*tan(x1*(_a4_**(x1**x1) + x1))**2
<lambdifygenerated-11571>:2: RuntimeWarning: invalid value encountered in power
  return x1*tan(x1*(_a4_**(_a3_**(_a5_*x1)) + x1))**2
<lambdifygenerated-11575>:2: RuntimeWarning: overflow encountered in power
  return x1*tan(x1*(_a4_**(_a3_**(_a5_*x1)) + x1**2))**2
<lambdifygenerated-11575>:2: RuntimeWarning: invalid valu

leaky_ReLU 0.0 1 0
hello
bye
leaky_ReLU 0.0 2 0
hello
bye
leaky_ReLU 0.0 3 0
hello


<lambdifygenerated-11641>:2: RuntimeWarning: invalid value encountered in power
  return x1**2/(2*x1 + cos(_a7_*x1**x1)) + x1
/export/home/shared/Projects/ANN/Sergio/BMS_approximator/bin/ann_functions/../no_degeneracy/mcmc.py:654: OptimizeWarning: Covariance of the parameters could not be estimated
  res = curve_fit(
<lambdifygenerated-11642>:2: RuntimeWarning: invalid value encountered in power
  return x1**2/(2*x1 + cos(_a7_*x1**x1)) + x1
<lambdifygenerated-11643>:2: RuntimeWarning: invalid value encountered in power
  return x1**2/(2*x1 + cos(_a0_**x1*_a7_)) + x1
<lambdifygenerated-11647>:2: RuntimeWarning: invalid value encountered in power
  return x1**2/(_a2_ + x1 + cos(_a0_**x1*_a7_)) + x1
<lambdifygenerated-11651>:2: RuntimeWarning: invalid value encountered in power
  return _a5_*x1/(_a2_ + x1 + cos(_a0_**x1*_a7_)) + x1


bye
leaky_ReLU 0.0 4 0
hello
bye
leaky_ReLU 0.0 5 0
hello


...

<lambdifygenerated-25417>:2: RuntimeWarning: invalid value encountered in power
  return x1**x1/x1
<lambdifygenerated-25418>:2: RuntimeWarning: invalid value encountered in power
  return x1**x1/x1


bye
tanh 0.2 0 2
hello
bye
tanh 0.2 1 2
hello
bye
tanh 0.2 2 2
hello
bye
tanh 0.2 3 2
hello
bye
tanh 0.2 4 2
hello
bye
tanh 0.2 5 2
hello
bye
tanh 0.2 6 2
hello


<lambdifygenerated-25505>:2: RuntimeWarning: invalid value encountered in power
  return tanh(x1**x1)
<lambdifygenerated-25506>:2: RuntimeWarning: invalid value encountered in power
  return tanh(x1**x1)
<lambdifygenerated-25507>:2: RuntimeWarning: invalid value encountered in power
  return tanh(_a3_**x1)
<lambdifygenerated-25513>:2: RuntimeWarning: invalid value encountered in power
  return tanh(_a3_**x1)
<lambdifygenerated-25525>:2: RuntimeWarning: invalid value encountered in power
  return exp(_a2_*x1**x1)
/export/home/shared/Projects/ANN/Sergio/BMS_approximator/bin/ann_functions/../no_degeneracy/mcmc.py:654: OptimizeWarning: Covariance of the parameters could not be estimated
  res = curve_fit(
<lambdifygenerated-25526>:2: RuntimeWarning: invalid value encountered in power
  return exp(_a2_*x1**x1)


bye
tanh 0.2 7 2
hello
bye
tanh 0.2 8 2
hello
bye
tanh 0.2 9 2
hello
bye


,sigma,function,mae_nn_train,mae_nn_test,mae_mdl_train,mae_mdl_test,rmse_nn_train,rmse_nn_test,rmse_mdl_train,rmse_mdl_test,n,r
0,0.0,leaky_ReLU,0.010827,0.117873,0.000799,0.032633,0.012699,0.132914,0.001039,0.045034,0,0
1,0.0,leaky_ReLU,0.006434,0.164714,0.007437,0.197682,0.009403,0.176908,0.010081,0.218711,1,0
2,0.0,leaky_ReLU,0.007582,0.140876,0.005103,0.175677,0.010593,0.170455,0.006735,0.206563,2,0
3,0.0,leaky_ReLU,0.003688,0.172617,0.001652,0.067233,0.005361,0.202771,0.002061,0.080441,3,0
4,0.0,leaky_ReLU,0.004480,0.105094,0.002559,1.141700,0.005718,0.128489,0.002954,1.636346,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...
655,0.2,tanh,0.058561,0.370645,0.025333,0.127387,0.101806,0.424306,0.040428,0.134413,5,2
656,0.2,tanh,0.101504,0.885691,0.083983,0.275630,0.144480,0.918737,0.118438,0.277328,6,2
657,0.2,tanh,0.085416,1.122517,0.094920,0.028182,0.116586,1.216812,0.116271,0.031750,7,2
658,0.2,tanh,0.105510,0.233941,0.059270,0.319420,0.145616,0.239003,0.077461,0.427289,8,2
